In [5]:
print("ram ram")

ram ram


In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from influxdb_client import InfluxDBClient

# ============================================================
# CONFIG
# ============================================================

load_dotenv()

IP = os.getenv("IIIOT_IP")
PORT = os.getenv("IIIOT_PORT", "8086")
TOKEN = os.getenv("INFLUXDB_TOKEN")
ORG = "IIIOT-INFOTECH"

OUTPUT_DIR = Path(
    r"C:\Users\Administrator\Desktop\MAI (19-08-26)\New folder\docs\iot_info"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

URL = f"http://{IP}:{PORT}"

print("=" * 70)
print("InfluxDB Schema + Backup Export")
print("=" * 70)
print(f"InfluxDB : {URL}")
print(f"Organization : {ORG}")
print(f"Output : {OUTPUT_DIR}")
print()


# ============================================================
# VALIDATION
# ============================================================

if not IP:
    raise ValueError("IIIOT_IP is missing from .env")

if not TOKEN:
    raise ValueError("INFLUXDB_TOKEN is missing from .env")


# ============================================================
# CONNECT
# ============================================================

client = InfluxDBClient(
    url=URL,
    token=TOKEN,
    org=ORG,
    timeout=120_000,
)

try:

    # --------------------------------------------------------
    # HEALTH CHECK
    # --------------------------------------------------------

    health = client.health()

    print(f"InfluxDB status : {health.status}")
    print(f"InfluxDB version: {health.version}")

    if health.status != "pass":
        raise RuntimeError("InfluxDB health check failed.")

    # --------------------------------------------------------
    # ORGANIZATION
    # --------------------------------------------------------

    org_api = client.organizations_api()
    bucket_api = client.buckets_api()
    query_api = client.query_api()

    organizations = org_api.find_organizations()

    org_info = []

    for org in organizations:
        org_info.append({
            "id": org.id,
            "name": org.name
        })

    # --------------------------------------------------------
    # BUCKETS
    # --------------------------------------------------------

    buckets_response = bucket_api.find_buckets()

    buckets = []

    for bucket in buckets_response.buckets:

        buckets.append({
            "id": bucket.id,
            "name": bucket.name,
            "org_id": bucket.org_id,
            "retention_rules": [
                {
                    "type": rule.type,
                    "every_seconds": rule.every_seconds,
                    "shard_group_duration_seconds":
                        rule.shard_group_duration_seconds
                }
                for rule in (bucket.retention_rules or [])
            ]
        })

    print()
    print(f"Buckets found: {len(buckets)}")

    for bucket in buckets:
        print(f"  - {bucket['name']}")

    # ========================================================
    # SCHEMA
    # ========================================================

    schema = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "influxdb": {
            "url": URL,
            "version": health.version,
            "organization": ORG
        },
        "organizations": org_info,
        "buckets": []
    }

    # ========================================================
    # BACKUP
    # ========================================================

    backup = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "influxdb": {
            "url": URL,
            "version": health.version,
            "organization": ORG
        },
        "buckets": []
    }

    # --------------------------------------------------------
    # PROCESS EACH BUCKET
    # --------------------------------------------------------

    for index, bucket in enumerate(buckets, start=1):

        bucket_name = bucket["name"]

        print()
        print(
            f"[{index}/{len(buckets)}] Processing bucket: "
            f"{bucket_name}"
        )

        bucket_schema = {
            "id": bucket["id"],
            "name": bucket_name,
            "measurements": []
        }

        bucket_backup = {
            "id": bucket["id"],
            "name": bucket_name,
            "measurements": {}
        }

        # ----------------------------------------------------
        # Get measurements
        # ----------------------------------------------------

        measurement_query = f'''
import "influxdata/influxdb/schema"

schema.measurements(
    bucket: "{bucket_name}"
)
'''

        try:
            measurement_tables = query_api.query(
                measurement_query,
                org=ORG
            )

            measurements = []

            for table in measurement_tables:
                for record in table.records:
                    value = record.get_value()

                    if value and value not in measurements:
                        measurements.append(value)

        except Exception as e:

            print(
                f"  ! Measurement discovery failed: {e}"
            )

            schema["buckets"].append(bucket_schema)
            backup["buckets"].append(bucket_backup)

            continue

        print(
            f"  Measurements found: {len(measurements)}"
        )

        # ----------------------------------------------------
        # Process measurements
        # ----------------------------------------------------

        for measurement in measurements:

            print(
                f"    -> {measurement}"
            )

            measurement_schema = {
                "name": measurement,
                "fields": [],
                "tags": []
            }

            # =================================================
            # FIELDS
            # =================================================

            field_query = f'''
import "influxdata/influxdb/schema"

schema.measurementFieldKeys(
    bucket: "{bucket_name}",
    measurement: "{measurement}"
)
'''

            try:

                field_tables = query_api.query(
                    field_query,
                    org=ORG
                )

                fields = []

                for table in field_tables:
                    for record in table.records:

                        value = record.get_value()

                        if value and value not in fields:
                            fields.append(value)

                measurement_schema["fields"] = fields

            except Exception as e:

                print(
                    f"       Field discovery failed: {e}"
                )

            # =================================================
            # TAGS
            # =================================================

            tag_query = f'''
import "influxdata/influxdb/schema"

schema.measurementTagKeys(
    bucket: "{bucket_name}",
    measurement: "{measurement}"
)
'''

            try:

                tag_tables = query_api.query(
                    tag_query,
                    org=ORG
                )

                tags = []

                for table in tag_tables:
                    for record in table.records:

                        value = record.get_value()

                        if value and value not in tags:
                            tags.append(value)

                measurement_schema["tags"] = tags

            except Exception as e:

                print(
                    f"       Tag discovery failed: {e}"
                )

            bucket_schema["measurements"].append(
                measurement_schema
            )

            # =================================================
            # DATA BACKUP
            # =================================================
            #
            # We retrieve data in chunks.
            #
            # Adjust BACKUP_RANGE if required.
            #

            BACKUP_RANGE = "-30d"

            data_query = f'''
from(bucket: "{bucket_name}")
    |> range(start: {BACKUP_RANGE})
    |> filter(fn: (r) =>
        r._measurement == "{measurement}"
    )
'''

            try:

                tables = query_api.query(
                    data_query,
                    org=ORG
                )

                records = []

                for table in tables:

                    for record in table.records:

                        data = record.values.copy()

                        # Remove internal objects that aren't
                        # necessary in JSON backup.

                        clean_data = {}

                        for key, value in data.items():

                            if isinstance(
                                value,
                                (str, int, float, bool)
                            ) or value is None:

                                clean_data[key] = value

                            else:

                                clean_data[key] = str(value)

                        records.append(clean_data)

                bucket_backup["measurements"][
                    measurement
                ] = records

                print(
                    f"       Data points: {len(records)}"
                )

            except Exception as e:

                print(
                    f"       Data backup failed: {e}"
                )

                bucket_backup["measurements"][
                    measurement
                ] = {
                    "error": str(e)
                }

        # ----------------------------------------------------
        # Add bucket results
        # ----------------------------------------------------

        schema["buckets"].append(bucket_schema)
        backup["buckets"].append(bucket_backup)

    # ========================================================
    # SAVE SCHEMA
    # ========================================================

    schema_file = OUTPUT_DIR / "influxdb_schema.json"

    with open(
        schema_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            schema,
            f,
            indent=4,
            ensure_ascii=False
        )

    # ========================================================
    # SAVE BACKUP
    # ========================================================

    backup_file = OUTPUT_DIR / "influxdb_backup.json"

    with open(
        backup_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            backup,
            f,
            indent=4,
            ensure_ascii=False
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("EXPORT COMPLETED")
    print("=" * 70)

    print()
    print("Schema:")
    print(schema_file)

    print()
    print("Backup:")
    print(backup_file)

    print()
    print("Nothing was deleted or modified in InfluxDB.")
    print()
    print("=" * 70)

finally:

    client.close()